# 14쪽 · 메모리가 있을 때

## 준비
아래 셀은 파일 경로와 `.env` 로더를 준비합니다. 한 번 실행하고 넘어가세요.

In [ ]:
from pathlib import Path
import sys

# 노트북을 연 위치가 ipynb 폴더이면 한 단계 위로 이동합니다.
ROOT = Path.cwd()

if ROOT.name == "ipynb":
    ROOT = ROOT.parent

if not (ROOT / "lab" / "config.py").exists():
    raise RuntimeError("Langgraph_6_Week_Code 또는 ipynb 폴더에서 실행하세요.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lab.config import make_llm, load_settings

## 1. 준비와 도구 함수

함수는 여기에서 직접 확인할 수 있습니다.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition


def add_numbers(a: int, b: int) -> int:
    """두 정수 a와 b를 더합니다."""
    return a + b


def multiply_numbers(a: int, b: int) -> int:
    """두 정수 a와 b를 곱합니다."""
    return a * b

## 2. 모델에 도구 연결

한 번의 응답에서 도구를 순차적으로 요청하도록 설정합니다.

In [ ]:
llm = make_llm()

tools = [add_numbers, multiply_numbers]
tool_llm = llm.bind_tools(
    tools,
    parallel_tool_calls=False,
)

## 3. Assistant 정의

도구 결과를 받은 뒤에도 같은 tool_llm을 사용합니다.

In [ ]:
system = SystemMessage(
    content=(
        "계산은 제공된 도구를 사용하세요. "
        "도구 결과를 확인하고 필요한 계산을 이어서 수행하세요. "
        "계산이 끝나면 한국어로 답하세요. "
        "이전 결과를 모르면 추측하지 말고 물어보세요."
    )
)


def assistant(state: MessagesState):
    messages = [system] + state["messages"]
    response = tool_llm.invoke(messages)

    return {"messages": [response]}

## 4. 그래프 연결

tools에서 assistant로 돌아오는 한 줄이 핵심입니다.

In [ ]:
builder = StateGraph(MessagesState)

builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)

# 도구 결과를 모델에게 돌려보냅니다.
builder.add_edge("tools", "assistant")

## 5. 메모리와 대화 번호

이 셀을 다시 실행하면 새 메모리로 시작합니다.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

config = {
    "configurable": {
        "thread_id": "1",
    },
}

### 그래프 보기

In [ ]:
from lab.display import show_mermaid

show_mermaid(graph)

## 6. 첫 번째 질문

thread 1에 첫 대화를 저장합니다.

In [ ]:
messages = [
    HumanMessage(content="3과 4를 더해줘."),
]

first = graph.invoke(
    {"messages": messages},
    config,
)

for message in first["messages"]:
    message.pretty_print()

## 7. 같은 대화에 두 번째 질문

config를 바꾸지 않고 사용합니다.

In [ ]:
messages = [
    HumanMessage(content="그 결과에 2를 곱해줘."),
]

second = graph.invoke(
    {"messages": messages},
    config,
)

for message in second["messages"]:
    message.pretty_print()

## 8. 저장된 사용자 메시지

이번에는 첫 질문도 보이는지 확인합니다.

In [ ]:
for message in second["messages"]:
    if isinstance(message, HumanMessage):
        print(message.content)